In [1]:
import sys
import os
import importlib
from pathlib import Path

project = Path("/mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo")
assert project.is_dir(), f"项目目录不存在：{project}"
os.chdir(project)

print("Python 版本：", sys.version.split()[0])
print("Python 路径：", sys.executable)
print("工作目录：", Path.cwd())

print("\n依赖检查：")
for name in [
    "numpy", "scipy", "pandas", "matplotlib",
    "yaml", "mendeleev", "tqdm",
    "flux_history", "mineral_utils",
]:
    try:
        module = importlib.import_module(name)
        print(f"[OK] {name}: {getattr(module, '__version__', '无版本字段')}")
        if name in ("flux_history", "mineral_utils"):
            print("     来源：", module.__file__)
    except Exception as exc:
        print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")

print("\n关键输入：")
for relative in [
    "Data/mineral_library.yaml",
    "Data/flux_data/Baseline_5.0GV_11.0km.npz",
    "Data/Geant4_data/Olivine/neutron/outNuclei_0.873326.txt",
    "Data/Geant4_data/Olivine/Range/DEDX_Z4_A4.txt",
]:
    path = project / relative
    print(f"[{'存在' if path.is_file() else '缺失'}] {relative}")

Python 版本： 3.10.12
Python 路径： /home/sjh/.venvs/primus-paleo/bin/python
工作目录： /mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo

依赖检查：
[OK] numpy: 2.2.6
[OK] scipy: 1.15.3
[OK] pandas: 2.3.3
[OK] matplotlib: 3.10.9
[OK] yaml: 6.0.3
[OK] mendeleev: 1.3.0
[OK] tqdm: 4.70.1
[OK] flux_history: 无版本字段
     来源： /mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo/flux_history.py
[OK] mineral_utils: 无版本字段
     来源： /mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo/mineral_utils.py

关键输入：
[存在] Data/mineral_library.yaml
[存在] Data/flux_data/Baseline_5.0GV_11.0km.npz
[存在] Data/Geant4_data/Olivine/neutron/outNuclei_0.873326.txt
[缺失] Data/Geant4_data/Olivine/Range/DEDX_Z4_A4.txt


In [2]:
import os
import shutil
import subprocess
from pathlib import Path
from mendeleev import element

geant_dir = Path("Data/Geant4_data").resolve()

print("Geant4 项目目录：", geant_dir)

for relative in [
    "Run.cc",
    "CMakeLists.txt",
    "build/Run",
    "build/CMakeCache.txt",
    "build/materials.mac",
]:
    path = geant_dir / relative
    print(f"[{'存在' if path.is_file() else '缺失'}] {relative}")

print("\n编译工具：")
for command in ["cmake", "c++", "geant4-config"]:
    executable = shutil.which(command)
    print(f"{command}: {executable or '当前 PATH 中未找到'}")

print("\nGeant4 配置：")
for key in ["Geant4_DIR", "CMAKE_PREFIX_PATH"]:
    print(f"{key}: {os.environ.get(key, '未设置')}")

config = shutil.which("geant4-config")
if config:
    result = subprocess.run(
        [config, "--version"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    print("Geant4 版本查询：", result.stdout.strip() or result.stderr.strip())

print("\nmendeleev 中的铍质量数：",
      sorted(iso.mass_number for iso in element("Be").isotopes))

Geant4 项目目录： /mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo/Data/Geant4_data
[存在] Run.cc
[存在] CMakeLists.txt
[缺失] build/Run
[缺失] build/CMakeCache.txt
[缺失] build/materials.mac

编译工具：
cmake: /usr/bin/cmake
c++: /usr/bin/c++
geant4-config: /usr/local/bin/geant4-config

Geant4 配置：
Geant4_DIR: 未设置
CMAKE_PREFIX_PATH: /mnt/e/WSL/home/sjh/root
Geant4 版本查询： 11.2.0

mendeleev 中的铍质量数： [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


In [3]:
from pathlib import Path
import numpy as np

path = Path(
    "/mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo/"
    "Data/Geant4_data/build/dedx_check/DEDX_Z4_A4.txt"
)

table = np.loadtxt(path, skiprows=1, ndmin=2)

assert table.shape[0] >= 2, "数据点不足"
assert table.shape[1] == 4, "数据应有四列"
assert np.isfinite(table).all(), "存在 NaN 或无穷值"

energy, dedx_e, dedx_n, track_range = table.T
event_energy = 19.73008635

checks = {
    "能量全部为正": np.all(energy > 0),
    "能量严格递增": np.all(np.diff(energy) > 0),
    "两类阻止本领均非负": np.all(dedx_e >= 0) and np.all(dedx_n >= 0),
    "总阻止本领全部为正": np.all(dedx_e + dedx_n > 0),
    "射程全部为正": np.all(track_range > 0),
    "射程严格递增": np.all(np.diff(track_range) > 0),
    "覆盖该 Be4 记录的动能": energy.min() <= event_energy <= energy.max(),
}

print("数据行数：", len(table))
print(f"能量范围：{energy.min():.6g} ～ {energy.max():.6g} MeV")
print(f"射程范围：{track_range.min():.6g} ～ {track_range.max():.6g} μm")

for name, passed in checks.items():
    print(f"[{'通过' if passed else '未通过'}] {name}")

print("\n最后五行：")
print(table[-5:])

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
数据行数： 149
能量范围：1e-05 ～ 9622.33 MeV
射程范围：0.00385152 ～ 295103 μm
[通过] 能量全部为正
[通过] 能量严格递增
[通过] 两类阻止本领均非负
[通过] 总阻止本领全部为正
[通过] 射程全部为正
[未通过] 射程严格递增
[通过] 覆盖该 Be4 记录的动能

最后五行：
[[5.501600e+03 9.227296e-03 0.000000e+00 2.951033e+05]
 [6.326840e+03 9.071510e-03 0.000000e+00 2.951033e+05]
 [7.275866e+03 8.959292e-03 0.000000e+00 2.951033e+05]
 [8.367246e+03 8.885388e-03 0.000000e+00 2.951033e+05]
 [9.622333e+03 8.844947e-03 0.000000e+00 2.951033e+05]]


In [4]:
import numpy as np

energy, dedx_e, dedx_n, track_range = table.T
delta_range = np.diff(track_range)

flat = np.flatnonzero(delta_range == 0)
decreasing = np.flatnonzero(delta_range < 0)
bad = np.flatnonzero(delta_range <= 0)

print("相邻射程相等的区间数：", len(flat))
print("射程下降的区间数：", len(decreasing))

if len(bad):
    i = int(bad[0])
    print("\n第一处不严格递增的位置：")
    print(f"能量：{energy[i]:.8g} → {energy[i+1]:.8g} MeV")
    print(f"射程：{track_range[i]:.8g} → {track_range[i+1]:.8g} μm")

    print("\n附近数据：")
    print("Energy_MeV  dEdx_Elec  dEdx_Nucl  Range_um")
    print(table[max(0, i-2):min(len(table), i+4)])

event_energy = 19.73008635
j = int(np.searchsorted(energy, event_energy))
assert 0 < j < len(energy), "事件能量没有被表内两个能量点包围"

print(f"\nBe4 记录动能：{event_energy} MeV")
print("包围该能量的两行：")
print(table[j-1:j+1])

print(
    "从最低能量到事件上侧网格点，射程是否严格递增：",
    bool(np.all(np.diff(track_range[:j+1]) > 0)),
)

相邻射程相等的区间数： 6
射程下降的区间数： 0

第一处不严格递增的位置：
能量：4160 → 4784 MeV
射程：295103.3 → 295103.3 μm

附近数据：
Energy_MeV  dEdx_Elec  dEdx_Nucl  Range_um
[[3.145558e+03 1.040031e-02 0.000000e+00 2.099855e+05]
 [3.617391e+03 1.001748e-02 0.000000e+00 2.563400e+05]
 [4.160000e+03 9.693516e-03 0.000000e+00 2.951033e+05]
 [4.784000e+03 9.432471e-03 0.000000e+00 2.951033e+05]
 [5.501600e+03 9.227296e-03 0.000000e+00 2.951033e+05]
 [6.326840e+03 9.071510e-03 0.000000e+00 2.951033e+05]]

Be4 记录动能：19.73008635 MeV
包围该能量的两行：
[[1.785984e+01 3.551730e-01 1.941351e-04 3.005864e+01]
 [2.053882e+01 3.202946e-01 1.714632e-04 3.804512e+01]]
从最低能量到事件上侧网格点，射程是否严格递增： True


In [5]:
from pathlib import Path
import numpy as np

root = Path(
    "/mnt/d/GFOPhmx/Research_work/Old_Muon/"
    "primus-paleo/Data/Geant4_data/build"
)

old = np.loadtxt(root / "dedx_check/DEDX_Z4_A4.txt", skiprows=1)
new = np.loadtxt(
    root / "dedx_check_csda10GeV/DEDX_Z4_A4.txt", skiprows=1
)

assert old.shape == new.shape, "两次表格大小不同"
assert np.array_equal(old[:, 0], new[:, 0]), "两次能量网格不同"

print("新表数值全部有限：", np.isfinite(new).all())
print("新表射程全部为正：", np.all(new[:, 3] > 0))
print("新表射程严格递增：", np.all(np.diff(new[:, 3]) > 0))
print("新表射程相等区间数：", np.sum(np.diff(new[:, 3]) == 0))

low = old[:, 0] <= 25.0
relative_change = np.abs(new[low, 3] / old[low, 3] - 1)
print("≤25 MeV 射程最大相对变化：", relative_change.max())

print("\n新表最后五行：")
print(new[-5:])

新表数值全部有限： True
新表射程全部为正： True
新表射程严格递增： True
新表射程相等区间数： 0
≤25 MeV 射程最大相对变化： 0.0

新表最后五行：
[[5.501600e+03 9.227296e-03 0.000000e+00 4.541593e+05]
 [6.326840e+03 9.071510e-03 0.000000e+00 5.446076e+05]
 [7.275866e+03 8.959292e-03 0.000000e+00 6.501559e+05]
 [8.367246e+03 8.885388e-03 0.000000e+00 7.727874e+05]
 [9.622333e+03 8.844947e-03 0.000000e+00 9.147069e+05]]


In [6]:
from pathlib import Path
import numpy as np

root = Path(
    "/mnt/d/GFOPhmx/Research_work/Old_Muon/"
    "primus-paleo/Data/Geant4_data"
)

src = root / "build/dedx_check_csda10GeV/DEDX_Z4_A4.txt"
dst = root / "Olivine/Range/DEDX_Z4_A4.txt"

# 再检查一次即将使用的文件
data = np.loadtxt(src, skiprows=1, ndmin=2)
assert data.shape[1] == 4
assert np.isfinite(data).all()
assert np.all(data[:, 3] > 0)
assert np.all(np.diff(data[:, 0]) > 0)
assert np.all(np.diff(data[:, 3]) > 0)

content = src.read_bytes()

# 重复运行时不覆盖不同内容的文件
if dst.exists():
    assert dst.read_bytes() == content, "目标已有不同内容，停止覆盖"
    print("目标文件已存在，且与测试表一致")
else:
    with dst.open("xb") as f:
        f.write(content)
    print("已复制测试表")

assert dst.read_bytes() == content
print("目标路径：", dst)
print("数据行数：", len(data))
print("文件一致性检查：通过")

已复制测试表
目标路径： /mnt/d/GFOPhmx/Research_work/Old_Muon/primus-paleo/Data/Geant4_data/Olivine/Range/DEDX_Z4_A4.txt
数据行数： 149
文件一致性检查：通过


In [1]:
import sys
print(sys.executable)

/home/sjh/.venvs/primus-paleo/bin/python
